# zero-grad-set-none — ex1: implement zero_grad with set_to_none semantics

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `zero-grad-set-none`. Running the final beacon cell reports progress against the `PyTorch: zero_grad` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: zero_grad` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`zero-grad-set-none`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "zero-grad-set-none"
DD_SUBTOPIC = "PyTorch: zero_grad"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `optimizer.zero_grad(set_to_none=True)` — quick refresher

PyTorch accumulates gradients into `param.grad` on every `.backward()`. Without an explicit reset, the gradient from batch N would be ADDED to the one from batch N+1, producing a stale, oversized update. `optimizer.zero_grad()` is what makes mini-batch SGD actually behave like stochastic gradient descent.

**`set_to_none=True` (default since PyTorch 1.7) vs `False`.** `set_to_none=True` replaces `.grad` with `None`; the next `.backward()` allocates a fresh gradient tensor. `set_to_none=False` keeps the buffer and writes zeros into it. `None` is faster (no kernel launch, no memory to clear) and lets autograd skip the addition in the next backward, but downstream code that reads `param.grad` must handle the `None` case.

### Exercise 1 — implement zero_grad with set_to_none semantics

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the `set_to_none=True` convention by iterating a parameter list and setting each `.grad` attribute to `None`, matching PyTorch's default zero_grad behavior.
> Keywords: zero-grad, set-to-none, hand-rolled-optimizer
> ```

**KCs targeted:** `zero-grad-set-to-none-semantics`, `zero-grad-iterates-params-list`

Implement `ex1_zero_grad(params)`. This is the body of a hand-rolled optimizer's `.zero_grad()` method, using the modern `set_to_none=True` convention.

1. Iterate `params` (a list of `nn.Parameter` / leaf tensors with `requires_grad=True`).
2. For each `p`, set `p.grad = None`.
3. Do NOT use `.detach()`, do NOT use `torch.zeros_like`, do NOT call `.zero_()` on existing grad buffers. The point is the None convention.

After calling `ex1_zero_grad(params)` every parameter must have `p.grad is None`. After the NEXT `.backward()`, PyTorch will allocate a fresh `.grad` tensor automatically.

No return value — the function mutates the params in place.

In [ ]:
def ex1_zero_grad(params):
    for p in params:
        p.grad = None


<details><summary>Solution</summary>

```python
def ex1_zero_grad(params):
    for p in params:
        p.grad = None
```

**Why the modern convention is `None`, not zeros.** Before PyTorch 1.7 the default was `set_to_none=False` — `.grad` was kept and zeroed. Two reasons it changed:

1. Memory: a `None` grad releases the tensor; the next backward allocates fresh and PyTorch's autograd can sometimes avoid creating it at all (e.g. parameters not used in the current batch).
2. Speed: `p.grad = None` is a pointer assignment (free). `p.grad.zero_()` launches a CUDA kernel.

**Downstream consequence.** Code that reads `p.grad` AFTER a `zero_grad(set_to_none=True)` and BEFORE the next backward must handle `None` — typical pattern is `if p.grad is not None: ...`. Optimizers internally do exactly this in their `.step()` loop.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()